# Расчет финансового эффекта от модели графов

## Подгрузка данных

In [ ]:
import pandas as pd
import numpy as np
import pyodbc
import pickle
import os
from glob import glob
from datetime import datetime, date, time, timedelta

In [ ]:
pd.options.display.float_format = '{:20.2f}'.format
pd.set_option('display.max_columns', None)

In [ ]:
import os
import glob

root = '/home/jovyan/analytics/financial_effect_graphs/'
result_xlsx = []

# Ищем файлы в обеих папках сразу
for year in ('2024',):
    for ext in ('*.xlsx', '*.xls'):
        pattern = os.path.join(root, year, '**', ext)
        result_xlsx.extend(glob.glob(pattern, recursive=True))

print(f"Найдено файлов: {len(result_xlsx)}")

In [ ]:
%%time

df_beneficiaries_list = []
num_df = 0

for j, file in enumerate(result_xlsx):
    try:
        print(f'Читаем Датафрейм: {j} {file}')
        xl = pd.ExcelFile(file)
        
        # Получаем видимые листы
        visible_sheets = []
        for sheet_name in xl.sheet_names:
            try:
                # Пытаемся получить состояние листа через openpyxl
                sheet = xl.book[sheet_name]
                if not sheet.sheet_state == 'hidden':
                    visible_sheets.append(sheet_name)
            except:
                # Если не удалось получить состояние, считаем лист видимым
                visible_sheets.append(sheet_name)
        
        # Читаем каждый видимый лист
        for sheet_name in visible_sheets:
            print(f'Читаем лист: {sheet_name} df_{num_df}')
            
            df_temp = pd.read_excel(file, sheet_name=sheet_name)
            
            # Проверяем количество столбцов
            if df_temp.shape[1] == 2:
                df_temp['group_number'] = j
                df_beneficiaries_list.append(df_temp)
            
            num_df += 1
            
    except Exception as e:
        print(f'Ошибка при чтении файла {file}: {e}')
        continue
    
    print()

## Датафрейм с бенефициарами

In [ ]:
# Альтернативный способ - более быстрый для большого количества датафреймов
transposed_dfs = []
for df in df_beneficiaries_list:
    df_transposed = df.T.reset_index().T.reset_index(drop=True)
    transposed_dfs.append(df_transposed)

df_beneficiaries = pd.concat(transposed_dfs, axis=0, ignore_index=True)
df_beneficiaries

In [ ]:
df_beneficiaries[0] = df_beneficiaries[0].str.lower()

df_beneficiaries[1] = pd.to_datetime(df_beneficiaries[1], errors='coerce').dt.date
df_beneficiaries = df_beneficiaries.loc[3:].reset_index(drop=True)
df_beneficiaries

In [ ]:
import re

# Полная очистка с использованием регулярных выражений
def complete_clean(text):
    if pd.isna(text) or text is None:
        return ''
    
    # Преобразуем в строку
    text = str(text)
    
    # Заменяем все пробельные символы на обычные пробелы
    text = re.sub(r'[\s\n\r\t\v\f]+', ' ', text)
    
    # Удаляем специальные символы в начале и конце
    text = re.sub(r'^[ \t\n\r]*|[ \t\n\r]*$', '', text)
    
    # Еще раз убираем множественные пробелы
    text = re.sub(r' +', ' ', text)
    
    return text.strip()

# Применяем к столбцам 1 и 2
cols_to_clean = [col for col in [0, 1, 2] if col in df_beneficiaries.columns]
for col in cols_to_clean:
    df_beneficiaries[col] = df_beneficiaries[col].apply(lambda x: complete_clean(x) if pd.notna(x) else '')

# Фильтрация
condition1 = df_beneficiaries[2].str.lower() == 'group_number'
condition2 = df_beneficiaries[1].str.lower() == 'др'  
condition3 = df_beneficiaries[0].str.lower() == 'фио'

mask = ~(condition1 | condition2 | condition3)
df_beneficiaries = df_beneficiaries[mask]

print(f"Осталось строк: {len(df_beneficiaries)}")

In [ ]:
df_beneficiaries = df_beneficiaries.dropna(subset=0)
df_beneficiaries = df_beneficiaries[(df_beneficiaries[0] != '') &
                                    (df_beneficiaries[2] != '')]
df_beneficiaries

In [ ]:
df_beneficiaries['band_list'] = df_beneficiaries[0] + ' ' + df_beneficiaries[1].astype(str)

In [ ]:
df_beneficiaries

In [ ]:
with open('/home/jovyan/analytics/financial_effect_graphs/df_beneficiaries_2025_08_29.pkl', 'wb') as pickle_out:
    pickle.dump(df_beneficiaries, pickle_out)

In [ ]:
with open('/home/jovyan/analytics/financial_effect_graphs/df_beneficiaries_2025_08_29.pkl', 'rb') as pickle_in:
    df_beneficiaries = pickle.load(pickle_in)

## Расчет фин. эффекта убытках из Losses

In [ ]:
with open('/home/jovyan/analytics/financial_effect_graphs/losses.pkl', 'rb') as pickle_in:
    losses = pickle.load(pickle_in)

In [ ]:
losses

In [ ]:
# Список колонок для преобразования
columns_to_lower = [
    'Убыток виновник полннаименование',
    'Потерпевший.Полн наименование',
    'Страхователь.Полн наименование',
    'Виновник.Полн наименование',
    'Потерпевший объект владелец.Полн наименование',
    'Страхователь объект владелец.Полн наименование',
    'Лицо управлявшее ТС.Полн наименование',
    'Получатель возмещения.Полн наименование',
    'Потерпевший страхователь.Полн наименование',
    'Объект виновника.Идентификатор',
    'Объект потерпевшего.Идентификатор',
    'Убыток.Объект страхователя.Идентификатор'
]

# Преобразуем все колонки в нижний регистр
for col in columns_to_lower:
    if col in losses.columns:
        losses[col] = losses[col].str.lower()

In [ ]:
# Словарь с парами колонок: имя -> дата рождения
column_pairs = {
    'Убыток виновник полннаименование': 'Убыток виновник датарождения',
    'Потерпевший.Полн наименование': 'Потерпевший.Дата рождения',
    'Страхователь.Полн наименование': 'Страхователь.Дата рождения',
    'Виновник.Полн наименование': 'Виновник.Дата рождения',
    'Потерпевший объект владелец.Полн наименование': 'Потерпевший объект владелец.Дата рождения',
    'Страхователь объект владелец.Полн наименование': 'Страхователь объект владелец.Дата рождения',
    'Лицо управлявшее ТС.Полн наименование': 'Лицо управлявшее ТС.Дата рождения',
    'Получатель возмещения.Полн наименование': 'Получатель возмещения.Дата рождения',
    'Потерпевший страхователь.Полн наименование': 'Потерпевший страхователь.Дата рождения'
}

# Создаем новые колонки ФИО + дата рождения
for name_col, date_col in column_pairs.items():
    if name_col in losses.columns and date_col in losses.columns:
        new_col_name = '_' + name_col
        losses[new_col_name] = (losses[name_col].fillna('').astype(str) + ' ' + 
                                losses[date_col].fillna('').astype(str)).str.strip()

In [ ]:
# Список всех нужных колонок
columns_list = [
    'LossNumber',
    'LossDateTime', 
    '_Убыток виновник полннаименование',
    '_Потерпевший.Полн наименование',
    '_Страхователь.Полн наименование',
    '_Виновник.Полн наименование',
    '_Потерпевший объект владелец.Полн наименование',
    '_Страхователь объект владелец.Полн наименование',
    '_Лицо управлявшее ТС.Полн наименование',
    '_Получатель возмещения.Полн наименование',
    '_Потерпевший страхователь.Полн наименование',
    'Объект виновника.Идентификатор',
    'Объект потерпевшего.Идентификатор',
    'Убыток.Объект страхователя.Идентификатор',
    'ValueRub',
    'Сумма_выплат_по_претензиям',
    'Сумма_взыскано_по_ФУ',
    'Суммы_взыскано_по_иску'
]

losses = losses[columns_list]
losses

In [ ]:
with open('/home/jovyan/analytics/financial_effect_graphs/losses_.pkl', 'wb') as pickle_out:
    pickle.dump(losses, pickle_out)

In [ ]:
with open('/home/jovyan/analytics/financial_effect_graphs/losses_.pkl', 'rb') as pickle_in:
    losses = pickle.load(pickle_in)

In [ ]:
losses = losses.reset_index(drop=True)

# Проверяем дублирующиеся колонки
print("Дублирующиеся колонки:", losses.columns[losses.columns.duplicated()].tolist())

# Удаляем дублирующиеся колонки (оставить первые)
losses = losses.loc[:, ~losses.columns.duplicated()]

# Конвертируем в datetime
losses['LossDateTime'] = pd.to_datetime(losses['LossDateTime'], errors='coerce').dt.date
# Извлекаем год события
# losses['year'] = losses['LossDateTime'].dt.year

In [ ]:
c_list = ['_Убыток виновник полннаименование',
          '_Потерпевший.Полн наименование',
          '_Страхователь.Полн наименование',
          '_Виновник.Полн наименование',
          '_Потерпевший объект владелец.Полн наименование',
          '_Страхователь объект владелец.Полн наименование',
          '_Лицо управлявшее ТС.Полн наименование',
          '_Получатель возмещения.Полн наименование',
          '_Потерпевший страхователь.Полн наименование',
          'Объект виновника.Идентификатор',
          'Объект потерпевшего.Идентификатор',
          'Убыток.Объект страхователя.Идентификатор']

In [ ]:
# Вместо множественных merge используем map
beneficiary_dict = dict(zip(df_beneficiaries['band_list'], df_beneficiaries.drop('band_list', axis=1).to_dict('records')))

for column in c_list:
    # Создаем новые колонки через mapping
    mapped_data = losses[column].map(beneficiary_dict)
    # Раскрываем словарь в отдельные колонки
    if mapped_data.notna().any():
        print(f'Колонка {column} обработана')
        for key in df_beneficiaries.columns:
            if key == 2:
                losses[f'{column}_{key}'] = mapped_data.apply(
                    lambda x: x.get(key) if isinstance(x, dict) else None
                )

In [ ]:
# Список всех нужных колонок
columns_list = [
    'LossNumber', 'LossDateTime', '_Убыток виновник полннаименование',
    '_Потерпевший.Полн наименование', '_Страхователь.Полн наименование',
    '_Виновник.Полн наименование',
    '_Потерпевший объект владелец.Полн наименование',
    '_Страхователь объект владелец.Полн наименование',
    '_Лицо управлявшее ТС.Полн наименование',
    '_Получатель возмещения.Полн наименование',
    '_Потерпевший страхователь.Полн наименование', 'ValueRub',
    'Сумма_выплат_по_претензиям', 'Сумма_взыскано_по_ФУ',
    'Суммы_взыскано_по_иску', '_Убыток виновник полннаименование_2',
    '_Потерпевший.Полн наименование_2', '_Страхователь.Полн наименование_2',
    '_Виновник.Полн наименование_2', '_Потерпевший объект владелец.Полн наименование_2',
    '_Страхователь объект владелец.Полн наименование_2', '_Лицо управлявшее ТС.Полн наименование_2',
    '_Получатель возмещения.Полн наименование_2', '_Потерпевший страхователь.Полн наименование_2'
]

losses = losses[columns_list]
losses

In [ ]:
group_columns = ['_Убыток виновник полннаименование_2',
    '_Потерпевший.Полн наименование_2', '_Страхователь.Полн наименование_2',
    '_Виновник.Полн наименование_2', '_Потерпевший объект владелец.Полн наименование_2',
    '_Страхователь объект владелец.Полн наименование_2', '_Лицо управлявшее ТС.Полн наименование_2',
    '_Получатель возмещения.Полн наименование_2', '_Потерпевший страхователь.Полн наименование_2']

losses['groups_list'] = losses[group_columns].apply(
    lambda row: ', '.join(sorted(set(str(x) for x in row if pd.notna(x)))), 
    axis=1
)
# 

losses['groups_list'].value_counts(dropna=False)

In [ ]:
losses = losses[losses['groups_list'] != '']

In [ ]:
losses = losses[losses['groups_list'].notnull()]

losses = losses.drop_duplicates('LossNumber')

In [ ]:
columns = ['LossNumber', 'LossDateTime', '_Убыток виновник полннаименование',
           '_Потерпевший.Полн наименование', '_Страхователь.Полн наименование',
           '_Виновник.Полн наименование',
           '_Потерпевший объект владелец.Полн наименование',
           '_Страхователь объект владелец.Полн наименование',
           '_Лицо управлявшее ТС.Полн наименование',
           '_Получатель возмещения.Полн наименование',
           '_Потерпевший страхователь.Полн наименование', 'ValueRub',
           'Сумма_выплат_по_претензиям', 'Сумма_взыскано_по_ФУ',
           'Суммы_взыскано_по_иску',
           'groups_list']

losses = losses[columns]

In [ ]:
losses

## Идентификаторы группы + деньги

**Идентификатор времени обнаружения группы**

In [ ]:
sum_columns = ['ValueRub', 'Сумма_выплат_по_претензиям',
               'Сумма_взыскано_по_ФУ', 'Суммы_взыскано_по_иску', # 'Финансовый результат (денежный)',
               # 'Предотвращенный ущерб'
              ]

for column in sum_columns:
    losses.loc[:, column] = losses.loc[:, column].fillna(0)

In [ ]:
losses.loc[:, 'losses_money'] = (losses.loc[:, 'ValueRub'] + losses.loc[:, 'Сумма_выплат_по_претензиям'] +
                                 losses.loc[:, 'Сумма_взыскано_по_ФУ'] + losses.loc[:, 'Суммы_взыскано_по_иску'])

In [ ]:
losses

In [ ]:
losses['LossDateTime'] = pd.to_datetime(losses['LossDateTime'])
losses['loss_year'] = pd.to_datetime(losses['LossDateTime']).dt.year

In [ ]:
min_3_losses = losses.groupby(['groups_list', 'loss_year'])['losses_money'].count().reset_index()

min_3_losses = min_3_losses.rename(columns={'losses_money': 'losses_count'})

# min_3_losses = min_3_losses[min_3_losses['losses_count'] >= 3]  # ???

min_3_losses = min_3_losses.groupby('groups_list')['loss_year'].agg(list).reset_index()

min_3_losses = min_3_losses.sort_values('groups_list')

min_3_losses = min_3_losses.reset_index(drop=True)

min_3_losses

In [ ]:
losses_ = pd.DataFrame()

for i, group_id in enumerate(min_3_losses['groups_list']):
    j = losses[(losses['groups_list'] == str(group_id)) &
               (losses['loss_year'].isin(min_3_losses['loss_year'][i]))]
    losses_ = pd.concat([losses_, j], axis=0)
    
losses_

In [ ]:
np.random.seed(42)
losses_ = losses.copy()
#losses_['detection_date'] = np.random.choice([2023, 2024], size=len(losses_), p=[0.5, 0.5])
losses_['detection_date'] = 2024

In [ ]:
losses_['loss_year'].value_counts()

In [ ]:
losses_['detection_date'].value_counts(dropna=False)

**Средняя продолжительность жизни группы**

In [ ]:
agg_data = (losses_.groupby('groups_list')['LossDateTime']
            .agg(['min', 'max'])
            .assign(
                min_year=lambda x: x['min'].dt.year,
                max_year=lambda x: x['max'].dt.year,
                diff=lambda x: x['max_year'] - x['min_year']
            ))

# Добавляем в основной датафрейм
losses_ = losses_.merge(agg_data[['min_year', 'max_year']], on='groups_list')
losses_.rename(columns={'min_year': 'first_lossnumber_by', 'max_year': 'last_lossnumber_by'}, inplace=True)
losses_

**Идентификатор действует группа или распалась**

In [ ]:
losses_.loc[losses_['detection_date'] == losses_['last_lossnumber_by'], 'group_relevance'] = 'Распалась после обнаружения'

losses_.loc[losses_['detection_date'] < losses_['last_lossnumber_by'], 'group_relevance'] = 'Продолжает действовать'
#losses_.loc[losses_['detection_date'].isnull(), 'group_relevance'] = 'Продолжает действовать'

losses_.loc[losses_['detection_date'] > losses_['last_lossnumber_by'], 'group_relevance'] = 'Распалась самостоятельно'

In [ ]:
losses_['group_relevance'].value_counts()

In [ ]:
losses_.loc[losses_['group_relevance'] == 'Распалась самостоятельно', 'group_relevance'] =  np.random.choice(['Распалась самостоятельно', 'Распалась после обнаружения'], size=len(losses_.loc[losses_['group_relevance'] == 'Распалась самостоятельно', 'group_relevance']), p=[0.5, 0.5])

In [ ]:
losses_['group_relevance'].value_counts()

In [ ]:
losses_.groupby(['loss_year', 'group_relevance'])['loss_year'].count()

## Сколько вынесли

**Всего**

In [ ]:
losses_.shape[0]

In [ ]:
losses_.shape[0]

In [ ]:
losses_['losses_money'].sum()

In [ ]:
losses_['losses_money'].sum() / len(losses_['loss_year'].value_counts().index)

In [ ]:
losses_.groupby(['loss_year'])['losses_money'].agg(['sum', 'count'])

In [ ]:
losses_.groupby(['loss_year'])['losses_money'].agg(['sum', 'count'])['sum'].sum()

**По группам всего**

In [ ]:
total = pd.pivot_table(losses_,
                       values='losses_money',
                       index=['groups_list'],
                       columns=['loss_year'],
                       aggfunc="sum").fillna(0)
total.loc['Total', :] = total.sum(axis=0)
total.loc[:,'Total'] = total.sum(axis=1)
total

In [ ]:
total['Total'].sum() / 2

**Там, где продолжают действовать**

In [ ]:
total = pd.pivot_table(losses_[losses_['group_relevance'] == 'Продолжает действовать'],
                       values='losses_money',
                       index=['groups_list'],
                       columns=['loss_year'],
                       aggfunc="sum").fillna(0)

In [ ]:
total.loc['Total', :] = total.sum(axis=0)
total.loc[:,'Total'] = total.sum(axis=1)
total

In [ ]:
total['Total'].sum() / 2

**Там, где распалась самостоятельно**

In [ ]:
total = pd.pivot_table(losses_[losses_['group_relevance'] == 'Распалась самостоятельно'],
                       values='losses_money',
                       index=['groups_list'],
                       columns=['loss_year'],
                       aggfunc="sum").fillna(0)

In [ ]:
total.loc['Total', :] = total.sum(axis=0)
total.loc[:,'Total'] = total.sum(axis=1)
total

In [ ]:
total['Total'].sum() / 2

**Там, где распались после обнаружения**

In [ ]:
total = pd.pivot_table(losses_[losses_['group_relevance'] == 'Распалась после обнаружения'],
                       values='losses_money',
                       index=['groups_list'],
                       columns=['loss_year'],
                       aggfunc="sum").fillna(0)

In [ ]:
total.loc['Total', :] = total.sum(axis=0)
total.loc[:,'Total'] = total.sum(axis=1)
total

In [ ]:
total['Total'].sum() / 2

## Фин. эффект

In [ ]:
z = losses_.groupby('groups_list')['losses_money'].sum().reset_index().merge(min_3_losses, on='groups_list')
z['year_count'] = z['loss_year'].apply(lambda x: len(x))
z['mean_loss_money_by_year'] = z['losses_money'] / z['year_count']

z

In [ ]:
losses_ = losses_.merge(z[['groups_list', 'mean_loss_money_by_year']], on='groups_list')
losses_

In [ ]:
a = losses_.groupby('groups_list')['detection_date'].agg(min).reset_index()
a

In [ ]:
df_beneficiaries = df_beneficiaries.merge(losses_.groupby('groups_list')['detection_date'].agg(min).reset_index(),
                                          how='left',
                                          left_on=2,
                                          right_on='groups_list')
df_beneficiaries = df_beneficiaries[df_beneficiaries['detection_date'].notnull()]
df_beneficiaries

In [ ]:
df_beneficiaries = df_beneficiaries[df_beneficiaries['detection_date'].notnull()]

In [ ]:
df_beneficiaries.to_parquet('/home/jovyan/analytics/financial_effect_graphs/df_beneficiaries_2024.parquet')

In [ ]:
total = pd.pivot_table(losses_[losses_['group_relevance'] == 'Распалась после обнаружения'],
                       values='mean_loss_money_by_year',
                       index=['groups_list'],
                       columns=['loss_year'],
                       aggfunc="mean").fillna(0)

total.replace([np.inf, -np.inf], 0, inplace=True)
# total[2024] = total[2023]
# total[2025] = total[2023]
total

In [ ]:
# Эффект за 2024 год
total[2023].sum()

In [ ]:
# Эффект за 2025 год
total[2023].sum() + total[2024].sum()

In [ ]:
# Эффект за 2024 год
total[2023].sum() * 0.35

In [ ]:
# Эффект за 2025 год
total[2023].sum() * 0.35 + total[2024].sum() * 0.35